# Denoising Diffusion Model 

One intiuitive reason for diffusion model to work well: making small, gradual changes is easier than makning one big leap. Directly mapping pure noise to a detailed image in one step is difficult, but diffusion breaks the generation into many small steps, each step only needs to add a little bit of detail or remove a little bit of noise. 

Transforming a noisy image into a slightly less noisy image is a much simpler task than generating an image from scratch. By chaining together many such small denoising steps, the model can achieve a complex transformation overall. In fact, this process is conceptually similar to the idea of a *denoising autoencoder*, where we train a network to clean up a corrupted input [1]. Here we just apply that idea iteratively, where the progressively denoised image can be viewed as latent representations.    


Each approach has its pros and cons, but diffusion models stood out for breaking the generation process into many small, reversible steps. Diffusion models are incremental updates from one distibution to another. Below we break down the transformation procedure into two main parts, forward process and backward process. 


## The Forward Diffussion Process 

The forward diffusion process is a fixed procedure that gradually adds randomness to the data. It is a sequence of small purturbations that eventually obliterate the original sample. Formally we define a sequence of latent variables $x_{1}, x_{2},... x_{T}$, where $x_{0}$ is a sample from an image in training data, and $x_{T}$ is pure noise after $T$ steps. At each step $t$ in the sequence, we add a bit of Gaussian noise: 

\begin{align}
    q \left( x_{t}|x_{t-1} \right) = \mathcal{N} \left( x_{t}; \sqrt{1-\beta_{t}}x_{t-1}, \beta_{t}\mathcal{I} \right)
\end{align}

where $0 < \beta_{t} < 1$ is a small noise level for step $t$. This is the key in forward process equation. It says that given the data at step $t-1$, we produce $x_{t}$ by taking most of $x_{t-1}$ and adding a little independent noise with variance $\beta_{t}$. Over many steps, these purturbations accumualte and the sample loses all identifiable features from the original. For a large number of steps $T$, $x_{T}$ approaches an isotropic Gaussian distrubution, essentially white noise. 

An important property of this defined process is we can describe the distribution of $x_{t}$ at any arbitrary step $t$ in closed form, as a function of the original data $x_{0}$. It turns out that $x_{t}$ is just a version of $x_{0}$ with heavy noise:

\begin{align}
    q \left( x_{t}|x_{0} \right) = \mathcal{N} \left( x_{t}; \sqrt{\bar{\alpha_{t}}}x_{0}, \left( 1 - \bar{\alpha_{t}} \right) \mathcal{I} \right)
\end{align}

where $\alpha_{t} := 1 - \beta_{t}$ and $\bar{\alpha_{t}} := \prod_{s=1}^{t}\alpha_{s}$. Intuitively $\sqrt{\bar{\alpha_{t}}}$ is the fraction of the original image remaining at time $t$, and $1-\bar{\alpha_{t}}$ is the fraction that has been replaced by noise. For example, if $\bar{\alpha_{t}} = 0.5$, then half of $x_{t}$ is still original image from $x_{0}$, and half is random noise. As $t$ grows, $\bar{\alpha_{t}}$ shrinks toward 0, meaning nearly all information is lost by the time we reach $x_{T}$. If we choose an appropriate schedule for $\beta_{t}$, by the final step $T$ the distribution $q \left( x_{T} \right)$ is essentially $\mathcal{N}(0,\mathcal{I})$, a standard normal distribution.    

It is worth noting the forward diffusion process is NOT a learning process, it is designed. We set the number of steps $T$ and schedule with noise levels $\beta_{t}$. For instance, a simple schedule can be increasing $\beta_{t}$ linearly from 0 over $T$ steps. This process serves as an encoder of transforming data into noise. 

## The Reverse Diffusion Process 

The reverse diffusion process is the generative model from traning. It aims to undo the forward process step by step. Ultimately, the model should learn to convert a noise sample into a synthetic data sample. If there was perfect knowledge of data distribution, each small reverse step would also be Gaussian. Specifically, one can show with Bayes rule that if $q(x_{t}|x_{t-1})$ is Gaussian and $\beta_{t}$ is small, then $q(x_{t-1}|x_{t})$ will also be a Gaussian distribution centered at some unknown mean depending on original data. However, we do not know the exact form because it would require knowing the true data distribution. 

We define a model $p_{\theta}$ for the reverse process ($\theta$ represent parameters): 

\begin{align}
    p_{\theta} \left( x_{0:T} \right) = p \left( x_{T} \right) \prod_{t=1}^{T} p_{\theta} \left( x_{t-1}|x_{t} \right)
\end{align}

with the learned reverse transition at each step modeled as a Gaussian:

\begin{align}
    p_{\theta} \left( x_{t-1}|x_{t} \right) = \mathcal{N} \left( x_{t-1}; \mu_{\theta} \left( x_{t}, t \right), \Sigma_{\theta} \left( x_{t}, t \right) \right)
\end{align}

Here $p(x_{T})$ is the prior distribution for the last step, and is set to equal $\mathcal{N}(x_{T};0,\mathcal{I})$, meaning the reverse chain will be started from pure Gaussian noise. The terms $\mu_{\theta}$ and $\Sigma_{\theta}$ are outputs of the network that take the current noisy sample $x_{t}$ and time step index $t$ as input, and predict the mean and variance of the distribution of the previous steps sample $x_{t-1}$. In simple terms, the model looks at noisy input and guesses what the less noisy version should look like. 

Typically implementations simplify the process. In practice we can fix or tie the variance $\Sigma_{\theta}$ to the same schedule as the forward process and focus on learning the mean [1]. For example, one common paramterization is to have the network predict the noise $\epsilon$ which is added at step $t$ then compute $\mu_{\theta}$ from the prediction. 

# Reference

- [1] https://lilianweng.github.io/posts/2021-07-11-diffusion-models/ 